## Concept 2 — ReAct Agent (Reason + Act) — Manual Implementation

Implements the original ReAct paper approach **without any agent framework**.
The LLM writes Thought/Action/Action Input as plain text. Your code parses it,
runs the tool, appends Observation, and loops — all written by hand.

### How all three notebooks differ
```
Notebook 1 — LLM + Tools (you manage the loop, LLM uses structured tool_calls):
  response = llm.bind_tools(tools).invoke(query)
  result   = tool.invoke(response.tool_calls[0]['args'])
  final    = llm.invoke(messages + [result])
  → YOU call each step. LLM returns JSON tool calls (easy to parse).

Notebook 2 — ReAct (you manage the loop, LLM writes text you parse):
  response = llm.invoke(react_prompt + scratchpad)   ← plain text output
  action, args = parse(response)                      ← fragile text parsing
  obs = tool.invoke(args)
  scratchpad += response + "Observation: " + obs
  → YOU still call each step. LLM writes "Thought/Action/Observation" as text.

Notebook 3 — create_agent (framework manages everything):
  agent = create_agent(model, tools)
  result = agent.invoke({'messages': [query]})
  → Framework runs the loop. Uses native tool_calls — no text parsing.
```

### What is ReAct?
ReAct (Reason + Act, 2022 paper): before each tool call, the LLM writes a **Thought**
explaining its reasoning. This makes every decision visible and debuggable.

### The loop you will write in this notebook
```
Question: query
  → [LLM writes]  Thought: what should I do?
  → [LLM writes]  Action: tool_name
  → [LLM writes]  Action Input: argument
  → [you parse]   extract tool name + argument from text
  → [you execute] observation = tool.invoke(argument)
  → [you append]  Observation: observation
  → repeat until LLM writes "Final Answer: ..."
```

### Why this is fragile (what Notebook 3 fixes)
The LLM must follow an exact text format. If it deviates even slightly,
`re.search()` fails and the loop breaks. Notebook 3 uses native `tool_calls`
(structured JSON) — no parsing, no fragility.

In [3]:
print('Hi')

Hi


In [1]:
import sys
sys.path.insert(0, '.')

from AgentUtils import get_llm, TOOLS, TEST_QUERIES, run_queries

# Plain LLM — NO .bind_tools() — forces text output instead of structured tool_calls.
# This is what makes ReAct different from Notebooks 1 and 3.
llm     = get_llm()
TOOL_MAP = {t.name: t for t in TOOLS}

print(f'LLM: plain text mode (no bound tools)')
print(f'Tools available: {list(TOOL_MAP.keys())}')

LLM: plain text mode (no bound tools)
Tools available: ['calculate', 'search_docs', 'get_weather']


### Step 1 — The ReAct Prompt

This prompt is the entire "framework". It tells the LLM:
- What tools exist (name + description)
- The EXACT text format to follow (Thought / Action / Action Input)
- How to finish (write "Final Answer:")

The LLM has no tools bound — it cannot call them itself.
YOUR code reads its text output, finds `Action:` and `Action Input:`, and runs the tool.

In [2]:
tools_description = '\n'.join(f'  {t.name}: {t.description}' for t in TOOLS)
tool_names        = ', '.join(TOOL_MAP.keys())

REACT_SYSTEM = f"""You are a helpful assistant that answers questions using tools.

Tools available:
{tools_description}

Reply using EXACTLY this format every time:

Thought: [why do I need to call a tool? what am I trying to find out?]
Action: [one of: {tool_names}]
Action Input: [the argument string for the tool]

After you get an Observation, continue with another Thought/Action or finish:

Thought: I now know the final answer
Final Answer: [your complete answer to the question]

IMPORTANT: Never skip the Thought line. Never call a tool not in the list above.
Do NOT write "Observation:" yourself — that will be added by the system."""

print(REACT_SYSTEM)

You are a helpful assistant that answers questions using tools.

Tools available:
  calculate: Evaluate a mathematical expression. Input must be a valid Python math expression e.g. '144 / 12'.
  search_docs: Search the company documentation for information about LangChain, agents, RAG, memory, tools, and AI topics.
  get_weather: Get the current weather for a given city.

Reply using EXACTLY this format every time:

Thought: [why do I need to call a tool? what am I trying to find out?]
Action: [one of: calculate, search_docs, get_weather]
Action Input: [the argument string for the tool]

After you get an Observation, continue with another Thought/Action or finish:

Thought: I now know the final answer
Final Answer: [your complete answer to the question]

IMPORTANT: Never skip the Thought line. Never call a tool not in the list above.
Do NOT write "Observation:" yourself — that will be added by the system.


### Step 2 — The Manual ReAct Loop

No AgentExecutor. No create_agent. Pure Python.

Each iteration:
1. Send (system prompt + scratchpad) to LLM → get plain text back
2. Parse `Action:` and `Action Input:` using regex
3. Execute the tool from TOOL_MAP
4. Append `Observation: result` to scratchpad
5. Repeat until the LLM writes `Final Answer:`

The `scratchpad` grows like this each iteration:
```
Question: What is 144 / 12?
Thought: I need to calculate 144 / 12
Action: calculate
Action Input: 144 / 12
Observation: 12.0
Thought: I now know the final answer
Final Answer: 144 divided by 12 is 12.
```

In [ ]:
import re
from langchain_core.messages import SystemMessage, HumanMessage

def run_react_loop(query: str, max_iterations: int = 6) -> str:
    scratchpad = f'Question: {query}\n'

    for step in range(1, max_iterations + 1):

        # ── 1. REASON: ask LLM what to do next ───────────────────────────────
        messages = [SystemMessage(content=REACT_SYSTEM), HumanMessage(content=scratchpad)]
        response = llm.invoke(messages).content.strip()
        print(f'[Step {step}]\n{response}\n')

        # ── 2. DONE? check for Final Answer ───────────────────────────────────
        if 'Final Answer:' in response:
            return response.split('Final Answer:', 1)[1].strip()

        # ── 3. ACT: parse Action and Action Input from text ───────────────────
        action_m = re.search(r'Action:\s*(\w+)', response)
        input_m  = re.search(r'Action Input:\s*(.+)', response)

        if not action_m or not input_m:
            # This is the fragility of text-format ReAct — if LLM deviates, it breaks
            print('[parse error] LLM did not follow Thought/Action/Action Input format.')
            return 'Could not parse agent response.'

        action       = action_m.group(1).strip()
        action_input = input_m.group(1).strip()

        # ── 4. OBSERVE: run the tool ──────────────────────────────────────────
        if action not in TOOL_MAP:
            observation = f'Unknown tool "{action}". Choose from: {list(TOOL_MAP.keys())}'
        else:
            observation = str(TOOL_MAP[action].invoke(action_input))

        print(f'Observation: {observation}\n')

        # ── 5. GROW scratchpad — feed observation back for next step ──────────
        scratchpad += response + f'\nObservation: {observation}\n'

    return 'Reached max iterations without a final answer.'

print('run_react_loop() ready — no framework, pure Python ReAct loop.')

### Step 3 — Watch the Full Thought → Action → Observation Loop

One tool needed. You will see the LLM write a Thought, name an Action,
YOUR code run the tool, append the Observation, then the LLM write Final Answer.

In [ ]:
answer = run_react_loop('What is 144 divided by 12?')
print(f'═══ Final Answer: {answer} ═══')

### Step 4 — Multi-Step: Two Tools in One Query

The loop runs TWICE. Each Observation is appended to the scratchpad so the
LLM can see what already happened before deciding the next action.

In [ ]:
answer = run_react_loop('Search docs for RAG and also calculate 25 multiplied by 4')
print(f'═══ Final Answer: {answer} ═══')

### Step 5 — No Tool Needed

The LLM can write `Final Answer:` on the very first step if no tool is required.
The loop still works — it just exits immediately after Step 1.

In [ ]:
answer = run_react_loop('What does LCEL stand for in LangChain?')
print(f'═══ Final Answer: {answer} ═══')

### Full Function + What Notebook 3 Fixes

`react_agent()` is a thin wrapper. All the real work is in `run_react_loop()`.

**What Notebook 3 (create_agent) replaces:**
- No regex parsing — LLM returns structured `tool_calls` JSON, not text
- No manual loop — the framework runs it
- No format-deviation risk — structured output cannot be misformatted
- Same result, 5 lines of code instead of 40

In [ ]:
def react_agent(query: str) -> str:
    return run_react_loop(query)

### Test — Standard Queries

In [ ]:
run_queries(react_agent, label='Concept 2 — ReAct Agent')